<a href="https://colab.research.google.com/github/Deepakdj007/daily-ai-mini-projects/blob/main/Fine_Tune_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

Thu May 14 02:24:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install unsloth
!pip install datasets trl transformers accelerate --quiet

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE          = None
LOAD_IN_4BIT   = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("✅ Model loaded successfully!")
print(f"   Model dtype: {model.dtype}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✅ Model loaded successfully!
   Model dtype: torch.float16


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


In [ ]:
from datasets import load_dataset

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token


def tokenize_examples(examples):
    """
    Formats and tokenises a batch of Alpaca examples in one step.

    Builds the full prompt string for each example, appends the EOS token,
    then runs the tokeniser to produce input_ids and attention_mask tensors.
    Returning these directly means the dataset already contains everything
    SFTTrainer's collator expects — no further preparation needed.

    Args:
        examples: A dict of lists (batched HuggingFace dataset format).

    Returns:
        A dict with 'input_ids' and 'attention_mask' lists.
    """
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = ALPACA_PROMPT.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)

    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

dataset = dataset.map(
    tokenize_examples,
    batched=True,
    remove_columns=dataset.column_names,
)

dataset = dataset.select(range(1000))

print(f"✅ Dataset ready — {len(dataset)} training examples")
print(f"   Columns: {dataset.column_names}")
print(f"   First example length: {len(dataset[0]['input_ids'])} tokens")

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

✅ Dataset ready — 1000 training examples
   Columns: ['input_ids', 'attention_mask']
   First example length: 203 tokens


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()

print("\n✅ Training complete!")
print(f"   Total training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"   Final loss: {trainer_stats.metrics['train_loss']:.4f}")

Unsloth: Switching to float32 training since model cannot work with float16
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,045,760 of 1,012,931,712 (1.29% trained)


Step,Training Loss
1,2.257785
2,2.766274
3,2.290232
4,2.511018
5,1.992951
6,1.856385
7,1.439927
8,1.406430
9,1.502338
10,1.308305


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.



✅ Training complete!
   Total training time: 612 seconds
   Final loss: 1.3321


In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = ALPACA_PROMPT.format(
    "Explain the concept of machine learning to a 10 year old.",
    "",
    "",
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 60)
print("FINE-TUNED MODEL OUTPUT:")
print("=" * 60)
response_section = response.split("### Response:")[-1].strip()
print(response_section)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINE-TUNED MODEL OUTPUT:
Machine learning is like teaching a computer to learn from examples, just like you learn from your parents, teachers, and friends. Instead of telling the computer exactly what to do, we give it lots of examples and let it figure out the rules on its own.

For example, if you want to teach a computer to recognize cats in pictures, you would show it thousands of pictures of cats and tell it "This is a cat!" and "That is not a cat!". Eventually, the computer would learn to recognize cats on its own, just like you do when you learn to recognize your friends.

There are different types of machine learning, but the most common one is called supervised learning. In supervised learning, we give the computer a set of examples with the correct answers, and the computer learns to recognize patterns in the data and make predictions.

In addition to supervised learning, there is also unsupervised learning, which is used to find patterns in the data without any labels. For e

In [ ]:
import os

ADAPTER_SAVE_PATH = "gemma-3-1b-lora-adapter"

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"✅ Adapter saved to: {ADAPTER_SAVE_PATH}/")
print("   Files in adapter directory:")
for f in os.listdir(ADAPTER_SAVE_PATH):
    size = os.path.getsize(f"{ADAPTER_SAVE_PATH}/{f}") / 1024 / 1024
    print(f"   {f}  ({size:.1f} MB)")

Unsloth: Restored added_tokens_decoder metadata in gemma-3-1b-lora-adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3-1b-lora-adapter.


✅ Adapter saved to: gemma-3-1b-lora-adapter/
   Files in adapter directory:
   tokenizer_config.json  (1.1 MB)
   README.md  (0.0 MB)
   adapter_model.safetensors  (49.8 MB)
   tokenizer.model  (4.5 MB)
   tokenizer.json  (31.8 MB)
   adapter_config.json  (0.0 MB)
   chat_template.jinja  (0.0 MB)


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("my-lora-adapter", "zip", ADAPTER_SAVE_PATH)
files.download("my-lora-adapter.zip")

print("✅ Download triggered — check your browser's Downloads folder")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download triggered — check your browser's Downloads folder


In [ ]:
import torch

def generate_response(instruction, context="", max_tokens=200):
    """
    Formats the instruction in Alpaca style, runs inference, and returns
    only the response section as a plain string.

    Args:
        instruction: The task description for the model.
        context:     Optional extra context (leave empty if not needed).
        max_tokens:  Maximum number of new tokens to generate.

    Returns:
        The generated response as a string, without the prompt prefix.
    """
    prompt = ALPACA_PROMPT.format(instruction, context, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("### Response:")[-1].strip()


test_cases = [
    {
        "instruction": "List 3 benefits of drinking water every morning.",
        "context": "",
    },
    {
        "instruction": "Write a Python function that checks if a number is prime.",
        "context": "",
    },
    {
        "instruction": "Summarise the following text in one sentence.",
        "context": (
            "The Eiffel Tower was built between 1887 and 1889 as the entrance arch "
            "for the 1889 World's Fair in Paris. It was designed by engineer Gustave "
            "Eiffel and has become one of the most recognisable structures in the world."
        ),
    },
]

for i, test in enumerate(test_cases, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}: {test['instruction'][:60]}...")
    if test["context"]:
        print(f"CONTEXT: {test['context'][:80]}...")
    print("─" * 60)
    print("RESPONSE:")
    print(generate_response(test["instruction"], test["context"]))

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEST 1: List 3 benefits of drinking water every morning....
────────────────────────────────────────────────────────────
RESPONSE:


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Improved energy levels: Drinking water throughout the day helps to maintain proper hydration, which can lead to increased energy levels and reduced fatigue.
2. Better digestion: Drinking water before meals can help to promote healthy digestion and prevent constipation.
3. Healthier skin: Drinking water can help to keep your skin hydrated and prevent it from looking dry and dull.

TEST 2: Write a Python function that checks if a number is prime....
────────────────────────────────────────────────────────────
RESPONSE:


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```python
def is_prime(n):
    """
    Checks if a number is prime.

    Args:
      n: An integer.

    Returns:
      True if the number is prime, False otherwise.
    """
    if n <= 1:
        return False

    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False

    return True
```

TEST 3: Summarise the following text in one sentence....
CONTEXT: The Eiffel Tower was built between 1887 and 1889 as the entrance arch for the 18...
────────────────────────────────────────────────────────────
RESPONSE:
The Eiffel Tower was built in 1889 as the entrance arch for the 1889 World's Fair in Paris and has become one of the most recognisable structures in the world.
